
# Day 49: Reinforcement Learning (RL) and RLHF

## Topics Covered
- Reinforcement Learning
- Policy Gradient
- Positive and Negative Rewards
- Random Guess → Reward → Learning
- RLHF Pipeline
- Pre-Training
- Alignment
- Supervised Fine-Tuning (SFT)
- Reward Models
- Pairwise Preference Learning
- Log-Sigmoid Loss
- Hands-on with GPT-2 and a Reward Model



# Theory

## Reinforcement Learning

In Reinforcement Learning (RL), an agent interacts with an environment and learns through rewards.

Agent → Action → Environment → Reward → Agent

The objective is to maximize cumulative reward.

### Positive Reward
Correct action:
Reward = +1

### Negative Reward
Incorrect action:
Reward = -1

### Random Guess

Initially, the model does not know the correct action and makes random guesses.

Over time:

Random Guess → Reward → Weight Update → Better Guess

Eventually the model learns an optimal policy.

## Policy

A policy defines the probability of taking an action given a state.

π(a|s)

## Policy Gradient

Policy Gradient methods directly optimize the policy.

θ = θ + α × Reward × ∇logπ(a|s)

Good actions receive positive rewards and become more likely.
Bad actions receive negative rewards and become less likely.

## RLHF Pipeline

Internet Data
↓
Pretraining
↓
Base GPT
↓
Supervised Fine-Tuning
↓
Reward Model
↓
PPO Reinforcement Learning
↓
Aligned ChatGPT

## Pre-Training

Objective:
Predict the next token.

Loss:
Cross Entropy Loss

## Alignment

Alignment means making AI systems:
- Helpful
- Honest
- Safe
- Harmless

## Supervised Fine-Tuning (SFT)

Humans create prompt-response pairs.

Prompt:
How do I learn Python?

Human Answer:
Start with variables, loops and functions.

This stage is expensive because humans must write responses manually.

## Reward Model

A separate model learns human preferences.

Input:
Prompt + Response

Output:
Reward Score

Example:

Response A:
Study Python and ML

Response B:
Eat pizza every day

Humans prefer A.

The reward model learns:

Reward(A) > Reward(B)

## How Does the Reward Model Learn?

Humans typically compare responses.

Prompt

Response A

Response B

Human selects:
A Better Than B

The reward model is trained on these preferences.

Humans usually do not assign numerical rewards directly.

## OpenAI Pairwise Preference Loss

L = -log(σ(r(chosen)-r(rejected)))

where:

σ(x)=1/(1+e^-x)

The objective is:

Preferred Response → Higher Reward
Rejected Response → Lower Reward

## Key Insight

The reward model does NOT learn:

'Which answer is correct?'

It learns:

'Which answer would humans prefer?'



# Hands-On 1: Load a Pretrained GPT Model

This example uses GPT-2 from Hugging Face.


In [1]:

from transformers import pipeline

generator = pipeline(
    "text-generation",
    model="gpt2"
)

prompt = "Explain Reinforcement Learning in simple terms:"

output = generator(
    prompt,
    max_new_tokens=80,
    do_sample=True,
    temperature=0.7
)

print(output[0]["generated_text"])


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

C:\Users\atanu\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\atanu\.cache\huggingface\hub\models--gpt2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.
Both `max_new_tokens` (=80) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Explain Reinforcement Learning in simple terms:

This is a very simple example for an understanding of reinforcement learning. I'm not sure the technical definition, but the general gist is that you can use reinforcement learning to model a situation that has been previously seen or predicted.

This is an example for an understanding of reinforcement learning. I'm not sure the technical definition, but the general gist is that you can use reinforcement learning to model



# Hands-On 2: Simulating Human Preference Data

Suppose humans prefer Response A over Response B.


In [2]:

prompt = "How do I learn AI?"

response_a = "Start with Python, machine learning and deep learning."
response_b = "Eat pizza every day."

chosen = response_a
rejected = response_b

print("Chosen:", chosen)
print("Rejected:", rejected)


Chosen: Start with Python, machine learning and deep learning.
Rejected: Eat pizza every day.



# Hands-On 3: Simple Reward Model

In production, reward models are transformer models.

Here we simulate one with a neural network.


In [3]:

import torch
import torch.nn as nn

class RewardModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(768, 1)

    def forward(self, x):
        return self.linear(x)

reward_model = RewardModel()

print(reward_model)


RewardModel(
  (linear): Linear(in_features=768, out_features=1, bias=True)
)



# Hands-On 4: OpenAI Pairwise Preference Loss


In [4]:

import torch
import torch.nn.functional as F

reward_chosen = torch.tensor([8.0])
reward_rejected = torch.tensor([3.0])

loss = -F.logsigmoid(
    reward_chosen - reward_rejected
)

print("Loss =", loss.item())


Loss = 0.006715348456054926



# Hands-On 5: Real Reward Model from Hugging Face

Many open-source reward models are available.

Example:

OpenAssistant Reward Model

This notebook demonstrates how such models are loaded.


In [5]:

#Uncomment when running locally

from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "OpenAssistant/reward-model-deberta-v3-large-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
reward_model = AutoModelForSequenceClassification.from_pretrained(model_name)

text = "Python is a great language for AI."

inputs = tokenizer(text, return_tensors="pt")

score = reward_model(**inputs).logits

print(score)


config.json:   0%|          | 0.00/993 [00:00<?, ?B/s]

C:\Users\atanu\AppData\Roaming\Python\Python313\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\atanu\.cache\huggingface\hub\models--OpenAssistant--reward-model-deberta-v3-large-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/455 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: OpenAssistant/reward-model-deberta-v3-large-v2
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tensor([[-2.5146]], grad_fn=<AddmmBackward0>)



# Mini Project

Build a miniature RLHF workflow:

1. Generate multiple GPT responses.
2. Rank responses manually.
3. Create chosen/rejected pairs.
4. Train a simple reward model.
5. Use reward scores to select the best response.

This mirrors the high-level RLHF pipeline used in modern LLMs.
